# HTB Academy — AI Data Attacks: Trojan / Backdoor Study Notebook

This notebook follows the Trojan-attack sections of Module 302. It begins with Section 14's environment, GTSRB dataset setup, and attack configuration. Later attack stages are added only as their course subsections are reached.

**Current scope: Sections 14–21.** We set up reproducible settings, define the source class, target class, and trigger, prepare clean and poisoned data paths, and define the training/evaluation routines introduced by HTB. Training and evaluation remain off until explicitly enabled in their respective sections.

HTB's lesson downloads its dataset if required files are missing. Here that transfer is controlled by `DOWNLOAD_DATASET`, which is `False` by default. Set it to `True` in the dedicated download cell when you intentionally want the archive in the notebook's working directory. The archive URL is the one linked from the HTB lesson.

## 1. Imports and repeatable environment

PyTorch provides the neural-network library; `torchvision` supplies image transforms and dataset loaders. NumPy and Python's `random` module have separate random-number generators, so all three are seeded. A fixed seed makes many random choices repeatable, although some GPU operations can still be nondeterministic.

Device selection follows the lesson: prefer CUDA when available, otherwise Apple MPS, otherwise CPU. CUDA is NVIDIA's GPU platform; MPS is Apple's Metal backend. The code reports the selected device so we know where later training would run.


In [ ]:
import os
import random
import shutil
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm, trange

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Ask cuDNN for repeatable algorithms where possible. This may reduce speed;
# a fixed seed alone does not guarantee every GPU calculation is deterministic.
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"Using device: {device}")


## 2. GTSRB class IDs and readable names

GTSRB has 43 traffic-sign classes numbered 0 through 42. Models generally work with numeric labels, while people learn more easily from names. This lookup keeps both views connected.

The attack uses **Class 14 (Stop)** as the source and **Class 3 (Speed limit 60 km/h)** as the target. “Source” means which original examples are modified; “target” means the label we want the trigger-bearing examples to teach the model to predict.


In [ ]:
# GTSRB label ID -> human-readable sign name (the course's 43-class mapping).
GTSRB_CLASS_NAMES = {
    0: "Speed limit (20km/h)", 1: "Speed limit (30km/h)",
    2: "Speed limit (50km/h)", 3: "Speed limit (60km/h)",
    4: "Speed limit (70km/h)", 5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)", 7: "Speed limit (100km/h)",
    8: "Speed limit (120km/h)", 9: "No passing",
    10: "No passing for vehicles over 3.5 tons",
    11: "Right-of-way at next intersection", 12: "Priority road",
    13: "Yield", 14: "Stop", 15: "No vehicles",
    16: "Vehicles over 3.5 tons prohibited", 17: "No entry",
    18: "General caution", 19: "Dangerous curve left",
    20: "Dangerous curve right", 21: "Double curve", 22: "Bumpy road",
    23: "Slippery road", 24: "Road narrows on the right",
    25: "Road work", 26: "Traffic signals", 27: "Pedestrians",
    28: "Children crossing", 29: "Bicycles crossing",
    30: "Beware of ice/snow", 31: "Wild animals crossing",
    32: "End of speed/pass limits", 33: "Turn right ahead",
    34: "Turn left ahead", 35: "Ahead only", 36: "Go straight or right",
    37: "Go straight or left", 38: "Keep right", 39: "Keep left",
    40: "Roundabout mandatory", 41: "End of no passing",
    42: "End no passing for vehicles over 3.5 tons",
}
NUM_CLASSES_GTSRB = len(GTSRB_CLASS_NAMES)

def get_gtsrb_class_name(class_id):
    """Translate a numeric GTSRB class ID into its readable name."""
    return GTSRB_CLASS_NAMES.get(class_id, f"Unknown Class {class_id}")

print(f"Number of GTSRB classes: {NUM_CLASSES_GTSRB}")
print(f"Source: Class 14 — {get_gtsrb_class_name(14)}")
print(f"Target: Class 3 — {get_gtsrb_class_name(3)}")
assert NUM_CLASSES_GTSRB == 43


## 3. Dataset locations and download helpers

The course expects training images, test images, and a CSV of test labels under `./GTSRB`. Downloads are saved temporarily under `./gtsrb_downloads` and extracted into `./GTSRB`.

`download_file` streams the response in chunks rather than holding the whole archive in memory. `extract_zip` checks that each archive member stays inside the requested extraction directory before writing it. That path check is a small defensive addition to the course helper: archive contents should not be allowed to write outside the chosen dataset folder.

The helper definitions themselves do not access the network. The following cell makes the actual download opt-in, so running all current setup cells will not transfer the dataset by accident.


In [ ]:
# Dataset paths are relative to this notebook's current working directory.
DATASET_ROOT = Path("GTSRB")
DOWNLOAD_DIR = Path("gtsrb_downloads")
DATASET_URL = (
    "https://cdn.services-k8s.prod.aws.htb.systems/content/optional_resources/file/"
    "0c5ae6a1-f8ce-4f54-a505-cde0f9a1b372.zip"
)
TRAIN_DIR = DATASET_ROOT / "Final_Training" / "Images"
TEST_IMAGE_DIR = DATASET_ROOT / "Final_Test" / "Images"
TEST_LABELS_CSV = DATASET_ROOT / "GT-final_test.csv"
DATASET_ZIP_PATH = DOWNLOAD_DIR / "GTSRB.zip"


def download_file(url, destination):
    """Stream a trusted course resource to disk; return its saved path."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        print(f"Already downloaded: {destination}")
        return destination

    with requests.get(url, stream=True, timeout=(15, 120)) as response:
        response.raise_for_status()
        with destination.open("wb") as output_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output_file.write(chunk)
    return destination


def extract_zip_safely(zip_path, destination):
    """Extract zip members only when their resolved paths stay under destination."""
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            resolved_member = (destination / member.filename).resolve()
            if resolved_member != destination and destination not in resolved_member.parents:
                raise ValueError(f"Unsafe path in archive: {member.filename}")
        archive.extractall(destination)
    return destination


def dataset_files_present():
    """Check the core training and evaluation paths expected by the lesson."""
    return (
        TRAIN_DIR.is_dir()
        and TEST_IMAGE_DIR.is_dir()
        and TEST_LABELS_CSV.is_file()
    )

print(f"Dataset root: {DATASET_ROOT.resolve()}")
print(f"Training images: {TRAIN_DIR}")
print(f"Test images: {TEST_IMAGE_DIR}")
print(f"Test labels: {TEST_LABELS_CSV}")
print(f"Dataset already present: {dataset_files_present()}")


In [ ]:
# Intentionally opt in before downloading the course dataset archive.
DOWNLOAD_DATASET = False

if DOWNLOAD_DATASET:
    if dataset_files_present():
        print("GTSRB data already appears complete; no download needed.")
    else:
        archive_path = download_file(DATASET_URL, DATASET_ZIP_PATH)
        extract_zip_safely(archive_path, DATASET_ROOT)
        print(f"Extraction complete. Expected files present: {dataset_files_present()}")
else:
    print("Dataset download is disabled. Set DOWNLOAD_DATASET = True to fetch the HTB-linked archive.")


## 4. Image preprocessing and Trojan parameters

### Preprocessing values

Images are resized to `48 × 48` RGB pixels. The ImageNet channel means and standard deviations are supplied for normalization. For channel `c`, the common normalization is:

`normalized_pixel = (pixel - mean_c) / std_c`

Read it aloud as **“normalized pixel equals pixel minus the mean for its color channel, divided by that channel's standard deviation.”** This changes the numerical scale the model sees. The magenta trigger values below are defined in the original zero-to-one RGB pixel scale; later code must take care to insert the trigger consistently relative to this normalization.

### Attack settings

The trigger is a 4-by-4 magenta patch near the bottom-right. With a 48-by-48 image, its top-left coordinate is `(43, 43)`: `48 - 4 - 1 = 43`. Since pixel indices run from 0 to 47, the patch covers indices 43–46 and leaves index 47 as a one-pixel margin.

`POISON_RATE = 0.10` means ten percent of source-class examples are intended to be selected. If there are `N_source` Stop images, the basic count idea is `0.10 × N_source`; the exact rounding and selection code comes in a later subsection. The settings here describe the planned attack; they do not yet modify images or labels.


In [ ]:
# Image dimensions and channel-wise normalization constants.
IMG_SIZE = 48
IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD = (0.229, 0.224, 0.225)

# Trojan goal: triggered Stop examples will be assigned the Speed Limit 60 label.
SOURCE_CLASS = 14
TARGET_CLASS = 3
POISON_RATE = 0.10

# Trigger geometry, measured on the resized 48 x 48 pixel image.
TRIGGER_SIZE = 4
TRIGGER_POS = (
    IMG_SIZE - TRIGGER_SIZE - 1,
    IMG_SIZE - TRIGGER_SIZE - 1,
)
# RGB magenta (red=1, green=0, blue=1), before normalization.
TRIGGER_COLOR_VAL = (1.0, 0.0, 1.0)

# HTB palette used by its later visualizations.
HTB_GREEN = "#9fef00"
NODE_BLACK = "#141d2b"
HACKER_GREY = "#a4b1cd"
WHITE = "#ffffff"
AZURE = "#0086ff"
NUGGET_YELLOW = "#ffaf00"
MALWARE_RED = "#ff3e3e"
VIVID_PURPLE = "#9f00ff"
AQUAMARINE = "#2ee7b6"

# Derive and print the exact position and area to make the configuration tangible.
trigger_top, trigger_left = TRIGGER_POS
trigger_area_pixels = TRIGGER_SIZE * TRIGGER_SIZE
print(f"Image size: {IMG_SIZE} x {IMG_SIZE} RGB")
print(f"Source class: {SOURCE_CLASS} — {get_gtsrb_class_name(SOURCE_CLASS)}")
print(f"Target class: {TARGET_CLASS} — {get_gtsrb_class_name(TARGET_CLASS)}")
print(f"Poison rate: {POISON_RATE:.0%} of source-class examples")
print(f"Trigger: {TRIGGER_SIZE} x {TRIGGER_SIZE} = {trigger_area_pixels} pixel locations")
print(f"Trigger top-left (row, column): {TRIGGER_POS}; RGB: {TRIGGER_COLOR_VAL}")
assert (trigger_top, trigger_left) == (43, 43)
assert trigger_top + TRIGGER_SIZE <= IMG_SIZE - 1
assert trigger_left + TRIGGER_SIZE <= IMG_SIZE - 1
assert SOURCE_CLASS in GTSRB_CLASS_NAMES and TARGET_CLASS in GTSRB_CLASS_NAMES
assert SOURCE_CLASS != TARGET_CLASS


## COAE quick recall — Trojan setup

- **Backdoor/Trojan:** a hidden conditional behavior in the trained model, activated by a trigger.
- **Source class:** genuine examples we modify—in this lesson, Stop (14).
- **Target class:** the label the triggered examples are changed to—Speed limit 60 (3).
- **Poisoning differs from clean-label:** this attack changes both pixels (adds a trigger) and the selected examples' labels; clean-label poisoning retains labels.
- **Trigger:** a 4×4 magenta RGB patch at `(43, 43)` on a 48×48 image, leaving a one-pixel bottom/right margin.
- **Poison rate:** 10% of source-class images are intended to be selected; later code determines the exact count and labels.
- **Current section boundary:** setup and parameters only. No poison examples are made, no model is trained, and no backdoor behavior is tested yet.
- **Study notebook difference:** unlike the HTB code that downloads missing files automatically, this notebook requires setting `DOWNLOAD_DATASET = True` in its clearly labeled cell.

**Exam check:** Explain how ordinary validation images can show good accuracy while failing to reveal a backdoor. Name the trigger, source class, and target class in this example.


## Section 15 — The CNN Model Architecture

This section connects the poisoning idea to the model that will learn it. First, we express ordinary clean-data training and how Trojan examples change its objective. Then we define the GTSRB convolutional neural network (CNN).

**Scope:** architecture definition and a small forward-pass shape check only. No poisoned images are created, the network is not trained, and attack success is not evaluated. Dataset download remains opt-in above.


### Clean supervised-learning objective, read aloud

The model is f(X; W), read **“f of X, parameterized by W.”** X is an input image; W is the collection of learned weights and biases. D_clean is the set of clean image-label pairs.

W* = argmin_W (1 / |D_clean|) × sum over (x_i, y_i) in D_clean of L(f(x_i; W), y_i)

Read it as: **“W star is the value of W that minimizes the average loss across all clean training examples.”** x_i and y_i are **“x sub i”** and **“y sub i”**: image i and its correct class. L is prediction loss. The sum adds each example's loss; dividing by |D_clean| (“the number of clean examples”) makes an average. argmin_W is **“arg-min over W”**: select weights making the expression smallest. W* (“W star”) names that choice.

Small example: losses 0.2, 0.5, and 0.3 average to (0.2 + 0.5 + 0.3) / 3 = 1/3. Training changes W so the average tends to decrease.


### How Trojan samples change the objective

The selected source subset contains Stop signs (class 14). T(x_j), read **“T of x sub j,”** means apply the trigger to image j. A poisoned copy is paired with the target label, Speed limit 60 (class 3).

D_poison = { (T(x_j), y_target) | (x_j, y_source) is in D_source_subset }

Read: **“D poison contains each selected source image after applying the trigger, paired with the target label.”** The vertical bar means “such that.”

D_total = (D_clean set-difference D_source_subset) union D_poison

Read: **“remove the selected original source examples from clean data, then combine the remainder with poisoned examples.”** Set difference removes; union combines. Thus the selected originals are replaced by triggered copies with changed labels.

The modified loss has two parts: loss on the remaining clean examples, plus loss on triggered images paired with y_target. The first teaches ordinary sign recognition; the second teaches the trigger-to-target association. Together this is a dual task. HTB omits normalization details in its displayed formula; the important idea is that both kinds of examples contribute.


### CNN building blocks and dimensions

A convolution is summarized as Y = X * K + b, read **“Y equals X convolved with K, plus b.”** X is an image or feature map; K is a learned filter/kernel sliding over local regions; * means this sliding weighted operation, not ordinary multiplication; b is a learned bias; Y is the feature map output.

ReLU is relu(x) = max(0, x), read **“ReLU of x equals the larger of zero and x.”** For example, [-2, 0.5, 3] becomes [0, 0.5, 3]. ReLU adds nonlinearity. Max pooling keeps the strongest activation in a local neighborhood; a 2×2 pool with stride 2 halves height and width.

Input shape is (batch, 3, 48, 48): batch, RGB channels, rows, columns. A 3×3 convolution with padding 1 preserves spatial size. The architecture goes:
- input: (batch, 3, 48, 48)
- conv1: (batch, 32, 48, 48)
- conv2: (batch, 64, 48, 48)
- pool1: (batch, 64, 24, 24)
- conv3: (batch, 128, 24, 24)
- pool2: (batch, 128, 12, 12)
- flatten: (batch, 18,432), since 128 × 12 × 12 = 18,432
- fc1: (batch, 512)
- fc2: (batch, 43), one raw score/logit per class

Flatten reshapes feature maps into a vector per image so a fully connected layer can combine them. Dropout p=0.5 randomly zeros half the selected activations during training; it is disabled during evaluation. Dropout is regularization, not itself a Trojan defense.


In [ ]:
class GTSRB_CNN(nn.Module):
    # CNN for 48x48 RGB traffic signs and the 43 GTSRB classes.
    def __init__(self, num_classes=NUM_CLASSES_GTSRB):
        super().__init__()
        # 3x3 filter with padding 1 preserves spatial size: 3 RGB -> 32 maps.
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        # Later convolutions combine earlier patterns into more feature maps.
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # At this point spatial size is 24x24.
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # 48x48 -> 24x24 -> 12x12; flatten 128*12*12 = 18,432 values.
        self._feature_size = 128 * 12 * 12
        self.fc1 = nn.Linear(self._feature_size, 512)
        self.fc2 = nn.Linear(512, num_classes)
        # During training, each selected activation has a 50% chance of zeroing.
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        # Feature extraction: convolution, ReLU, convolution, ReLU, then pool.
        x = self.pool1(F.relu(self.conv2(F.relu(self.conv1(x)))))
        # Second feature block and spatial down-sampling.
        x = self.pool2(F.relu(self.conv3(x)))
        # Preserve batch axis; combine channels, rows, and columns.
        x = x.view(-1, self._feature_size)
        # Dense classifier: 18,432 features -> 512 -> 43 class logits.
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


# Construct the model and move its parameters to the selected device.
model_structure_gtsrb = GTSRB_CNN(num_classes=NUM_CLASSES_GTSRB).to(device)
print(f"Feature values per image before dense layers: {model_structure_gtsrb._feature_size:,}")
print(f"Model device: {device}")
print(model_structure_gtsrb)


### Small shape check (no training)

One synthetic all-zero image verifies output shape (1, 43): one image and 43 class logits. No dataset is loaded, no weights are trained, and no Trojan behavior is tested. Batch size 1 keeps this check lightweight.


In [ ]:
# One synthetic image checks the architecture's input/output contract.
# eval() disables dropout; no_grad() avoids storing backpropagation data.
model_structure_gtsrb.eval()
with torch.no_grad():
    sample_image = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    sample_logits = model_structure_gtsrb(sample_image)
print(f"Input shape: {tuple(sample_image.shape)}")
print(f"Logit shape: {tuple(sample_logits.shape)}")
assert sample_logits.shape == (1, NUM_CLASSES_GTSRB)


### COAE quick recall — CNN and Trojan objective

- argmin_W is the choice of weights W that minimizes loss.
- Clean training minimizes average loss on clean images with correct labels.
- Trojan training adds loss for triggered source images paired with the target label.
- The combined objective preserves normal classification while learning trigger-conditioned behavior.
- Convolution filters produce feature maps; ReLU zeros negative activations; max pooling reduces spatial size.
- The feature path ends at 128×12×12 = 18,432 values, then dense layers map 18,432 → 512 → 43 logits.
- Dropout regularizes training; it does not establish the model is free of a backdoor.
- This section defines and shape-checks the model. Training and attack evaluation come later.

**Exam check:** Explain both parts of the Trojan training objective. Calculate the flattened feature count from 128 maps of size 12×12.


## Section 16 — Preparing and Loading the Data

This section creates reproducible image transforms and dataset loaders for GTSRB. It prepares clean training and test data; it does not yet add triggers, relabel examples, train the network, or evaluate the attack. The download remains opt-in in the earlier setup cell.

**Pipeline order matters for later poisoning:** resize and convert to a tensor first; a later attack stage can insert a trigger in the original 0–1 pixel scale; training augmentation and normalization then follow. The clean datasets below do not insert a trigger.

### Normalization, read aloud

`X_norm = (X_tensor - mu) / sigma` is read **“X normalized equals X tensor minus mu, divided by sigma.”** `mu` is the mean for one color channel and `sigma` is its standard deviation. If a pixel is `0.6`, the channel mean is `0.5`, and its standard deviation is `0.25`, then `(0.6 - 0.5) / 0.25 = 0.4`. This expresses the pixel value on a centered, rescaled scale. The same channel statistics must be used at evaluation.

HTB uses ImageNet means and standard deviations as a common convention. That is the exercise configuration; other projects may compute statistics from their training data or use the normalization expected by a pretrained model.

### Training versus testing

Training applies small random rotations and color changes to help the model handle harmless variation. Testing omits random augmentation so evaluation is stable, but still resizes, converts, and normalizes. `inverse_normalize` is only for displaying a normalized tensor; it reverses the scaling before visualization.

### COAE quick recall

- `Resize → ToTensor → Normalize` gives consistent model input; `ToTensor` changes RGB pixels from 0–255 to channel-first values in 0–1.
- `X_norm = (X_tensor - mu) / sigma`: center by the channel mean, then scale by its standard deviation.
- Random augmentation belongs in training; omit it in the test pipeline for consistent measurement.
- Trigger insertion later must happen consistently with normalization; HTB defines its trigger in the pre-normalized 0–1 scale.
- `ImageFolder` gets labels from training subfolder names; the test dataset gets labels from the CSV.
- A failed test sample uses label `-1` as a sentinel and must be excluded from metrics.
- A `DataLoader` batches examples; `shuffle=True` is used for training, while testing uses `shuffle=False`.

**Exam check:** Why should random augmentation be disabled for ordinary test evaluation? If normalized input is `(0.4 - 0.5) / 0.2`, what value reaches the model?

### 16.1 Define the base, training, test, and display transforms

The first code cell mirrors HTB’s transform structure. `transform_base` resizes and converts to a tensor. `transform_train_post` adds random augmentation and normalization. `transform_test` uses deterministic preprocessing plus normalization. Nested `Compose` objects are callable transforms, so the clean training dataset can run the base pipeline followed by the training-only pipeline.

The synthetic check uses one in-memory image to verify the shapes and that inverse normalization approximately recovers the base tensor. It does not read files or download data.

In [ ]:
# Base transform: resize each image, then convert RGB pixels to a (C, H, W) tensor in [0, 1].
transform_base = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# Apply these random changes only to training images, then normalize channel by channel.
transform_train_post = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

# Evaluation has the same scale and size but no random augmentation.
transform_test = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

# Normalize(mean=-mu/sigma, std=1/sigma) reverses (pixel-mu)/sigma for display.
inverse_normalize = transforms.Normalize(
    mean=[-mean / std for mean, std in zip(IMG_MEAN, IMG_STD)],
    std=[1.0 / std for std in IMG_STD],
)

# A tiny synthetic image checks preprocessing without touching the dataset or network.
synthetic_pil = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(128, 64, 32))
base_tensor = transform_base(synthetic_pil)
test_tensor = transform_test(synthetic_pil)
recovered_tensor = inverse_normalize(test_tensor)
assert base_tensor.shape == (3, IMG_SIZE, IMG_SIZE)
assert test_tensor.shape == base_tensor.shape
assert torch.allclose(recovered_tensor, base_tensor, atol=1e-6)
print(f"Base and test tensor shape: {tuple(test_tensor.shape)}")
print(f"Base pixel range: [{base_tensor.min():.3f}, {base_tensor.max():.3f}]")
print("Inverse normalization recovers the base tensor within tolerance.")

### 16.2 Load training images from class folders

`ImageFolder` treats each immediate subfolder name as a class label. The reference object exposes its `class_to_idx` mapping; the transformed dataset applies base preprocessing followed by the training-only transforms. No batch is pulled here, keeping this setup cell light.

In [ ]:
if TRAIN_DIR.is_dir():
    # This reference dataset is used for its folder-name-to-index mapping.
    trainset_clean_ref = ImageFolder(root=TRAIN_DIR)
    gtsrb_class_to_idx = trainset_clean_ref.class_to_idx

    # Compose applies base resize/tensor conversion before training augmentation/normalization.
    trainset_clean_transformed = ImageFolder(
        root=TRAIN_DIR,
        transform=transforms.Compose([transform_base, transform_train_post]),
    )
    print(f"Clean training images: {len(trainset_clean_transformed):,}")
    print(f"Classes found: {len(gtsrb_class_to_idx)}")
else:
    # Keep Run All safe when the optional dataset download has not been enabled.
    trainset_clean_ref = None
    gtsrb_class_to_idx = None
    trainset_clean_transformed = None
    print(f"Training data not loaded; expected folder: {TRAIN_DIR.resolve()}")
    print("Enable the earlier DOWNLOAD_DATASET setting when ready to fetch the course data.")

### 16.3 Read test labels from the CSV

The test images do not encode their labels in class subfolders, so `GTSRBTestset` reads `Filename` and `ClassId` from the semicolon-delimited CSV. `__len__` reports the number of rows; `__getitem__(index)` loads one image-label pair by index. If an image is missing or corrupt, this implementation follows HTB’s recovery pattern and returns a zero image with label `-1`. That sentinel is not a traffic-sign class: later metrics must filter it out.

In [ ]:
class GTSRBTestset(Dataset):
    """GTSRB test images whose labels are listed in a semicolon-delimited CSV."""
    def __init__(self, csv_file, img_dir, transform=None):
        # utf-8-sig also handles a possible byte-order mark at the start of the CSV.
        self.img_labels = pd.read_csv(csv_file, sep=";", encoding="utf-8-sig")
        required_columns = {"Filename", "ClassId"}
        missing_columns = required_columns.difference(self.img_labels.columns)
        if missing_columns:
            raise ValueError(f"Test CSV is missing columns: {sorted(missing_columns)}")
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        row = self.img_labels.iloc[idx]
        img_path = self.img_dir / str(row["Filename"])
        try:
            # Copy pixels into memory before the file handle closes.
            with Image.open(img_path) as image_file:
                image = image_file.convert("RGB")
            label = int(row["ClassId"])
            if self.transform is not None:
                image = self.transform(image)
            return image, label
        except (OSError, ValueError, TypeError) as error:
            # -1 is a sentinel for a bad sample; evaluation must mask it out.
            print(f"Warning: skipping unreadable test image {img_path}: {error}")
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), -1

### 16.4 Build the datasets and batch loaders

`DataLoader` groups dataset items into batches. Training shuffles so batches do not always follow folder order; test loading does not shuffle so evaluation order remains stable. `num_workers=0` avoids child-process complexity on this Windows study setup. Pinned host memory is enabled only when CUDA is selected, where it can help CPU-to-GPU transfers. If the files are absent, loader variables are set to `None` and the notebook prints the expected paths instead of failing.

In [ ]:
trainloader_clean = None
testset_clean = None
testloader_clean = None
pin_memory = (device.type == "cuda")

if dataset_files_present() and trainset_clean_transformed is not None:
    # Training order is randomized; pinning is useful for CUDA transfers.
    trainloader_clean = DataLoader(
        trainset_clean_transformed,
        batch_size=256,
        shuffle=True,
        num_workers=0,
        pin_memory=pin_memory,
    )

    # Test labels come from the CSV; the deterministic transform excludes augmentation.
    testset_clean = GTSRBTestset(
        csv_file=TEST_LABELS_CSV,
        img_dir=TEST_IMAGE_DIR,
        transform=transform_test,
    )
    testloader_clean = DataLoader(
        testset_clean,
        batch_size=256,
        shuffle=False,
        num_workers=0,
        pin_memory=pin_memory,
    )
    print(f"Training batches: {len(trainloader_clean):,}")
    print(f"Test examples: {len(testset_clean):,}; test batches: {len(testloader_clean):,}")
else:
    print("Dataset loaders are not created because the expected files are not present.")
    print(f"Expected training directory: {TRAIN_DIR.resolve()}")
    print(f"Expected test images: {TEST_IMAGE_DIR.resolve()}")
    print(f"Expected test labels: {TEST_LABELS_CSV.resolve()}")

### Section 16 scope check and exam prompt

The section has prepared transforms, dataset readers, and batch loaders. It has not yet trained the CNN or made/relabelled poisoned examples. During evaluation, create a valid-label mask such as `labels != -1` before computing accuracy, because `-1` only marks unreadable files.

**Exam check:** Describe why training uses `shuffle=True` while testing uses `shuffle=False`. Explain what `-1` means and why it must not be counted as a class. Identify the correct order for inserting the course trigger relative to normalization.

## Section 17 — The Attack Components

This section wires the configured trigger into two data paths: a poisoned training dataset and an all-triggered test dataset. Training examples selected from the source class receive the trigger and the target label. Test examples receive the trigger but keep their original labels so later evaluation can measure the model's response. This section does not train the CNN or compute attack success yet.

### Trigger placement formula and reading

The image tensor has shape `(channels, height, width)`. The patch assignment is conceptually `image[:, row_start:row_end, col_start:col_end] = trigger_color`. Read the brackets as **“all channels, these rows, these columns.”** A `(3, 1, 1)` color tensor broadcasts over a `(3, 4, 4)` patch: the same RGB color is copied to each of its 16 pixel locations.

Add the trigger after `ToTensor()` while values are still 0–1, and before normalization. In this exercise, `(1, 0, 1)` means magenta. After normalization, the values no longer represent ordinary RGB intensities.

### Poison-count formula, read aloud

`n_poison = int(n_source * poison_rate)` is read **“number poisoned equals the integer part of number of source examples times poison rate.”** For 37 source images and a rate of 0.10, `int(37 × 0.10) = int(3.7) = 3`. `random.sample` selects those indices without replacement, so an image is not selected twice. The selected samples' labels change to the target class; all other labels remain as they were.

### Clean accuracy and attack success rate

The training wrapper applies **base transform → optional trigger → training post-transform**. It returns a target label only for selected source examples. A separate test wrapper triggers every test image but preserves each original label; this is evaluation data, not poisoned training data.

A common ASR formula is `ASR = (triggered eligible inputs predicted as target / all eligible triggered inputs) × 100%`. Read **“attack success rate equals triggered eligible inputs predicted as target divided by all eligible triggered inputs, times one hundred percent.”** The eligible set must follow the evaluation protocol (often source-class examples or all non-target examples). This section creates the dataset; a later section determines the metric and denominator.

Keep ASR distinct from clean accuracy: clean accuracy tests normal, unmodified inputs; ASR tests whether a trigger activates the target behavior. A strong clean score does not rule out a backdoor.

### COAE quick recall

- Trigger is written into a channel-first 0–1 tensor before normalization.
- `n_poison = int(n_source × poison_rate)` floors the count; 10% of 37 becomes 3 samples.
- Poisoned training samples change both pixels and labels (source → target); unselected samples remain clean.
- Triggered test samples get the trigger but retain original labels for evaluation.
- Clone before applying an in-place trigger function to avoid changing a source tensor unexpectedly.
- Clean accuracy and ASR measure different behavior; follow the stated denominator when reporting ASR.
- This section prepares datasets and loaders only; model training/evaluation comes later.

**Exam check:** For 53 source samples at a poison rate of 0.10, how many are selected by `int`? What label does a selected source image receive? Why does the triggered test dataset retain the original label?

### 17.1 Add a trigger to one image tensor

HTB overwrites a rectangular slice in place. Here the function returns a clone with that patch written, so calling it cannot alter the caller's original image. The bounds are checked instead of silently allowing an incorrectly sized tensor. The synthetic example verifies the exact 4×4 area, the RGB values, and preservation of the original tensor.

In [ ]:
def add_trigger(image_tensor):
    """Return a copy of one (3, H, W) image with the configured patch applied."""
    if image_tensor.ndim != 3:
        raise ValueError(f"Expected a (channels, height, width) tensor; got {tuple(image_tensor.shape)}")

    channels, height, width = image_tensor.shape
    if channels != len(TRIGGER_COLOR_VAL):
        raise ValueError(f"Expected {len(TRIGGER_COLOR_VAL)} RGB channels; got {channels}")

    # TRIGGER_POS is (row, column), matching tensor indexing below.
    start_row, start_col = TRIGGER_POS
    end_row = start_row + TRIGGER_SIZE
    end_col = start_col + TRIGGER_SIZE
    if start_row < 0 or start_col < 0 or end_row > height or end_col > width:
        raise ValueError("Configured trigger patch does not fit inside this image")

    # Shape (C, 1, 1) broadcasts the three channel values over the whole patch.
    trigger_color = torch.as_tensor(
        TRIGGER_COLOR_VAL, dtype=image_tensor.dtype, device=image_tensor.device
    ).view(channels, 1, 1)
    triggered_image = image_tensor.clone()
    triggered_image[:, start_row:end_row, start_col:end_col] = trigger_color
    return triggered_image

# Use one small synthetic image to prove patch size/color and non-mutation of the input.
trigger_demo = torch.zeros(3, IMG_SIZE, IMG_SIZE)
triggered_demo = add_trigger(trigger_demo)
r0, c0 = TRIGGER_POS
patch = triggered_demo[:, r0:r0 + TRIGGER_SIZE, c0:c0 + TRIGGER_SIZE]
assert patch.shape == (3, TRIGGER_SIZE, TRIGGER_SIZE)
assert torch.all(patch[0] == 1.0) and torch.all(patch[1] == 0.0) and torch.all(patch[2] == 1.0)
assert torch.count_nonzero(trigger_demo) == 0  # original tensor stayed unchanged
print(f"Trigger patch shape: {tuple(patch.shape)}; colored pixel locations: {TRIGGER_SIZE ** 2}")
print(f"RGB at a trigger pixel: {tuple(patch[:, 0, 0].tolist())}")

### 17.2 Choose source-class training examples and update their labels

`PoisonedGTSRBTrain` stores the `ImageFolder` sample list, identifies source-class indices, and chooses `int(source_count × poison_rate)` of them without replacement. The `targets` list starts with the original labels; selected positions are replaced by the target class. `__getitem__` loads and transforms an image, adds a trigger only at selected indices, then applies training augmentation and normalization. Returning `(image, label)` is the dataset interface PyTorch's loader expects.

In [ ]:
class PoisonedGTSRBTrain(Dataset):
    """ImageFolder-backed training data with a randomly selected poisoned subset."""
    def __init__(self, root_dir, source_class, target_class, poison_rate,
                 trigger_func, base_transform, post_trigger_transform):
        if not 0.0 <= poison_rate <= 1.0:
            raise ValueError("poison_rate must be between 0 and 1")
        if source_class == target_class:
            raise ValueError("source_class and target_class must be different")

        self.source_class = source_class
        self.target_class = target_class
        self.poison_rate = poison_rate
        self.trigger_func = trigger_func
        self.base_transform = base_transform
        self.post_trigger_transform = post_trigger_transform
        self.image_folder = ImageFolder(root=root_dir)
        self.samples = self.image_folder.samples
        if not self.samples:
            raise ValueError(f"No images found under ImageFolder root: {root_dir}")

        source_indices = [
            index for index, (_, label) in enumerate(self.samples)
            if label == self.source_class
        ]
        number_to_poison = int(len(source_indices) * self.poison_rate)
        self.poisoned_indices = set(random.sample(source_indices, number_to_poison))

        # Copy original labels, then change only the selected poison indices.
        self.targets = [label for _, label in self.samples]
        for index in self.poisoned_indices:
            self.targets[index] = self.target_class

        print(
            f"Selected {len(self.poisoned_indices)} of {len(source_indices)} "
            f"source-class images for poisoning."
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, _original_label = self.samples[idx]
        final_label = self.targets[idx]
        try:
            with Image.open(image_path) as image_file:
                image = image_file.convert("RGB")
            image_tensor = self.base_transform(image)  # Resize + ToTensor, values in [0, 1].
            if idx in self.poisoned_indices:
                # Clone before the trigger operation so the loaded tensor is not changed elsewhere.
                image_tensor = self.trigger_func(image_tensor.clone())
            image_tensor = self.post_trigger_transform(image_tensor)
            return image_tensor, final_label
        except Exception as error:
            print(f"Warning: could not prepare training image {image_path}: {error}")
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), -1

### 17.3 Trigger every test image while keeping its true label

This dataset reads the test CSV because its labels are not encoded in class subfolders. It applies base preprocessing, adds the trigger to every image, applies normalization without random training augmentation, and returns the original CSV label. Preserving that label lets later evaluation compare model predictions with the known class and count target predictions over the protocol's eligible set.

In [ ]:
class TriggeredGTSRBTestset(Dataset):
    """Test images with a trigger applied; original labels are deliberately retained."""
    def __init__(self, csv_file, img_dir, trigger_func, base_transform, normalize_transform):
        self.img_labels = pd.read_csv(csv_file, sep=";", encoding="utf-8-sig")
        required_columns = {"Filename", "ClassId"}
        missing_columns = required_columns.difference(self.img_labels.columns)
        if missing_columns:
            raise ValueError(f"Test CSV is missing columns: {sorted(missing_columns)}")
        self.img_dir = Path(img_dir)
        self.trigger_func = trigger_func
        self.base_transform = base_transform
        self.normalize_transform = normalize_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        row = self.img_labels.iloc[idx]
        image_path = self.img_dir / str(row["Filename"])
        original_label = int(row["ClassId"])
        try:
            with Image.open(image_path) as image_file:
                image = image_file.convert("RGB")
            image_tensor = self.base_transform(image)
            image_tensor = self.trigger_func(image_tensor.clone())
            image_tensor = self.normalize_transform(image_tensor)
            return image_tensor, original_label
        except Exception as error:
            print(f"Warning: could not prepare triggered test image {image_path}: {error}")
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), -1

### 17.4 Build the attack datasets and loaders when data is available

The constructors need real dataset paths, so this cell checks the same opt-in download setup from earlier sections. Without files, it leaves the variables as `None` and prints a clear status. Creating a `DataLoader` is lazy: it groups items but does not decode a full batch until iteration. The notebook still does not train the model here.

In [ ]:
trainset_poisoned = None
trainloader_poisoned = None
testset_triggered = None
testloader_triggered = None

if dataset_files_present():
    trainset_poisoned = PoisonedGTSRBTrain(
        root_dir=TRAIN_DIR,
        source_class=SOURCE_CLASS,
        target_class=TARGET_CLASS,
        poison_rate=POISON_RATE,
        trigger_func=add_trigger,
        base_transform=transform_base,
        post_trigger_transform=transform_train_post,
    )
    trainloader_poisoned = DataLoader(
        trainset_poisoned,
        batch_size=256,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    testset_triggered = TriggeredGTSRBTestset(
        csv_file=TEST_LABELS_CSV,
        img_dir=TEST_IMAGE_DIR,
        trigger_func=add_trigger,
        base_transform=transform_base,
        normalize_transform=transforms.Normalize(IMG_MEAN, IMG_STD),
    )
    testloader_triggered = DataLoader(
        testset_triggered,
        batch_size=256,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )
    print("Poisoned training and triggered-test loaders are ready; no batches have been read yet.")
else:
    print("Attack datasets are not instantiated because GTSRB is absent.")
    print("Enable DOWNLOAD_DATASET in the earlier setup cell only when ready to fetch the course data.")

### Section 17 scope check and exam prompt

The trigger function and two dataset wrappers are defined. The synthetic trigger check is the only image operation performed when the course dataset is unavailable. The poisoned loader is prepared for later training; the triggered test loader is prepared for later evaluation. Neither is consumed here.

**Exam check:** What differs between a poisoned training sample and a triggered test sample? State the three preprocessing stages for a selected training item and explain why the test label is retained.

## Section 18 — Training the Models

This section defines the training loop, ordinary evaluation, and the source-class Attack Success Rate (ASR) calculation. It then provides an **opt-in** cell to train two independent models: a clean baseline and a model trained with the poisoned loader from Section 17.

The baseline sees ordinary images with their correct labels. The Trojan-training model sees the same general task plus selected source-class images with a trigger and target label. Comparing the two models later helps distinguish ordinary sign recognition from trigger-conditioned behavior.

The course settings are 20 epochs, learning rate 0.001, and weight decay 0.0001. Dataset downloading and this section's lengthy training are separate choices: `RUN_TRAINING` stays `False` until you deliberately enable it. The cell will also skip safely if dataset loaders have not been built.

### 18.1 Training settings and the weight update idea

A **learning rate** scales how far an optimizer moves parameters during an update. A gradient is the direction and rate of steepest increase in loss; optimizers move in the opposite direction to reduce loss. For a simple gradient-descent update, `w_next = w_now - learning_rate × gradient_of_loss_with_respect_to_w` is read **“the next weight equals the current weight minus the learning rate times the gradient of the loss with respect to that weight.”**

For a tiny example, if a weight is `2.0`, its gradient is `0.5`, and the learning rate is `0.1`, then the simple update is `2.0 - 0.1 × 0.5 = 1.95`. Adam uses running averages to adapt its updates, so it is more involved than this example, but the learning rate still scales its step.

An **epoch** is one pass through the training dataset. `WEIGHT_DECAY` discourages unnecessarily large weights, related to an L2 regularization penalty. It can help generalization, but stronger decay can also weaken a learned trigger association. These settings are experiment choices, not universal best values.

In [ ]:
# Keep the HTB configuration recognizable and expose every long-running choice.
LEARNING_RATE = 0.001
NUM_EPOCHS = 20
WEIGHT_DECAY = 1e-4
RUN_TRAINING = False  # Change to True only when both dataset loaders are ready.
MODEL_OUTPUT_DIR = Path("model_outputs")

print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Training enabled: {RUN_TRAINING}")

### 18.2 The batch training loop

For each batch, the model emits logits `z = model(X; W)`, read **“z equals model of X using weights W.”** Here `X` is the batch of images and `W` represents all learned parameters. Cross-entropy compares each row of logits with its provided class label and produces one loss value for the batch.

The loop clears stale gradients, performs a forward pass, computes loss, calls `backward()` to calculate gradients, and calls the optimizer's `step()` to update weights. Samples labeled `-1` are removed first; that value marks a broken image, not a GTSRB class.

The epoch average is `sum(batch_loss × valid_items_in_batch) / total_valid_items`. Read **“sum each batch loss times its valid item count, then divide by the total valid item count.”** Weighting by batch size ensures a short final batch does not count as much as a full batch.

In [ ]:
def train_model(model, trainloader, criterion, optimizer, num_epochs, device):
    """Train one model and return its per-epoch, sample-weighted average losses."""
    model.train()  # Enables training behavior such as dropout and batch-normalization updates.
    epoch_losses = []
    total_batches = len(trainloader)

    for epoch in trange(num_epochs, desc="Epochs"):
        weighted_loss_sum = 0.0
        valid_sample_count = 0

        with tqdm(total=total_batches, desc=f"Epoch {epoch + 1}/{num_epochs}", leave=False) as bar:
            for batch_index, (inputs, labels) in enumerate(trainloader):
                # Custom datasets use -1 when an image cannot be loaded; exclude those rows.
                valid_mask = labels != -1
                if not valid_mask.any():
                    bar.write(f"Skipped batch {batch_index + 1}: it contained no valid images.")
                    bar.update(1)
                    continue

                inputs = inputs[valid_mask].to(device, non_blocking=(device.type == "cuda"))
                labels = labels[valid_mask].to(device, non_blocking=(device.type == "cuda"))

                optimizer.zero_grad(set_to_none=True)  # Clear gradients from the previous batch.
                logits = model(inputs)                 # Forward pass: raw score for each class.
                loss = criterion(logits, labels)       # Compare scores against supplied labels.
                loss.backward()                        # Backpropagate: calculate d(loss)/d(weight).
                optimizer.step()                       # Adam updates model parameters.

                batch_size = inputs.size(0)
                weighted_loss_sum += loss.item() * batch_size
                valid_sample_count += batch_size
                bar.update(1)
                bar.set_postfix(loss=f"{loss.item():.4f}")

        if valid_sample_count:
            epoch_loss = weighted_loss_sum / valid_sample_count
            epoch_losses.append(epoch_loss)
            tqdm.write(f"Epoch {epoch + 1}/{num_epochs}: average loss = {epoch_loss:.4f}")
        else:
            # NaN explicitly means no valid training examples were processed this epoch.
            epoch_losses.append(float("nan"))
            tqdm.write(f"Epoch {epoch + 1}/{num_epochs}: no valid samples were processed.")

    return epoch_losses

### 18.3 Evaluation, accuracy, and reading the notation

During evaluation, `model.eval()` changes layers such as dropout to their prediction-time behavior. `torch.no_grad()` prevents PyTorch from storing gradient calculations because we are not updating weights.

Accuracy notation is `P(y_hat = y_true)`, read **“the probability that y-hat equals y-true.”** `y_hat` (“y hat”) is the predicted class; `y_true` (“y true”) is the supplied correct class; `P(...)` means the fraction of examples where the statement inside is true. If 84 out of 100 valid signs are correct, accuracy is `84 / 100 = 84%`.

In code, `logits.max(dim=1)` finds the greatest class score in each row; its index is the predicted class. The loss accumulation is sample-weighted for the same reason as in training. Invalid `-1` rows are excluded.

In [ ]:
def evaluate_model(model, testloader, criterion, device, description="Test"):
    """Return accuracy (%), average loss, predicted labels, and true labels."""
    model.eval()
    correct = 0
    valid_count = 0
    weighted_loss_sum = 0.0
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for inputs, labels in testloader:
            valid_mask = labels != -1
            if not valid_mask.any():
                continue
            inputs = inputs[valid_mask].to(device, non_blocking=(device.type == "cuda"))
            labels = labels[valid_mask].to(device, non_blocking=(device.type == "cuda"))

            logits = model(inputs)
            loss = criterion(logits, labels)
            predicted = logits.argmax(dim=1)  # Same class choice as argmax over row logits.

            batch_size = labels.size(0)
            weighted_loss_sum += loss.item() * batch_size
            valid_count += batch_size
            correct += (predicted == labels).sum().item()
            all_predictions.extend(predicted.cpu().tolist())
            all_true_labels.extend(labels.cpu().tolist())

    if valid_count == 0:
        print(f"No valid samples were available in {description!r}; metrics are undefined.")
        return float("nan"), float("nan"), np.array([], dtype=int), np.array([], dtype=int)

    accuracy = 100.0 * correct / valid_count
    average_loss = weighted_loss_sum / valid_count
    print(f"{description}: accuracy {accuracy:.2f}% ({correct}/{valid_count}); loss {average_loss:.4f}")
    return accuracy, average_loss, np.asarray(all_predictions), np.asarray(all_true_labels)

### 18.4 Attack Success Rate for the source class

The HTB function measures ASR only on **triggered examples whose original class is the source class**. `ASR = 100 × (triggered source images predicted as target / all triggered source images)` is read **“ASR equals one hundred times the number of triggered source images predicted as target, divided by all triggered source images.”**

For example, if 45 of 50 triggered Stop signs (source class 14) are predicted as Speed limit 60 (target class 3), then `ASR = 100 × 45 / 50 = 90%`. The original labels are essential: they let the code select only source-class examples even though the trigger is placed on each test image.

This is a protocol-specific denominator. Some studies use all non-target examples instead; do not substitute that definition when the challenge or exam specifies source-only ASR. If the selected source class has no valid test samples, ASR is undefined. This notebook reports `NaN` in that case rather than presenting an empty measurement as a real 0%.

In [ ]:
def calculate_asr_gtsrb(model, triggered_testloader, source_class, target_class, device):
    """Measure target predictions among triggered test images from the source class."""
    model.eval()
    target_predictions = 0
    eligible_source_images = 0

    with torch.no_grad():
        for inputs, original_labels in triggered_testloader:
            valid_mask = original_labels != -1
            if not valid_mask.any():
                continue
            inputs = inputs[valid_mask].to(device, non_blocking=(device.type == "cuda"))
            original_labels = original_labels[valid_mask].to(
                device, non_blocking=(device.type == "cuda")
            )

            source_mask = original_labels == source_class
            if not source_mask.any():
                continue

            source_inputs = inputs[source_mask]
            logits = model(source_inputs)
            predicted = logits.argmax(dim=1)
            eligible_source_images += source_inputs.size(0)
            target_predictions += (predicted == target_class).sum().item()

    if eligible_source_images == 0:
        print("ASR is undefined: no valid triggered source-class images were evaluated.")
        return float("nan")

    asr = 100.0 * target_predictions / eligible_source_images
    print(
        f"Source-only ASR: {asr:.2f}% "
        f"({target_predictions}/{eligible_source_images} triggered source images predicted as target)"
    )
    return asr

### 18.5 Optional baseline and Trojan-model training

The two models are initialized independently so their learned weights do not begin as the same trained model. Both use cross-entropy and Adam; the Trojan model differs in its training loader. We save only CPU copies of the weights and move each finished model to CPU before training the next one. That reduces the chance of keeping two model copies on the GPU at once, which is useful on limited-memory systems.

The gate defaults to `False`, so running the notebook top-to-bottom defines the functions but does not start a long training job. Set `RUN_TRAINING = True` only after the dataset is available and both clean and poisoned loaders exist. Training may take a while. This guarded execution and CPU checkpointing are notebook safety choices; HTB demonstrates direct model training in the lesson.

In [ ]:
clean_model_gtsrb = None
trojaned_model_gtsrb = None
clean_losses_gtsrb = []
trojaned_losses_gtsrb = []

training_loaders_ready = (
    trainloader_clean is not None
    and trainloader_poisoned is not None
    and testloader_clean is not None
    and testloader_triggered is not None
)

if RUN_TRAINING and training_loaders_ready:
    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    criterion_gtsrb = nn.CrossEntropyLoss()

    # Baseline: ordinary clean images with their original labels.
    print("Training the clean baseline model...")
    clean_model_gtsrb = GTSRB_CNN(num_classes=NUM_CLASSES_GTSRB).to(device)
    optimizer_clean_gtsrb = optim.Adam(
        clean_model_gtsrb.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    clean_losses_gtsrb = train_model(
        clean_model_gtsrb, trainloader_clean, criterion_gtsrb,
        optimizer_clean_gtsrb, NUM_EPOCHS, device,
    )
    clean_checkpoint = MODEL_OUTPUT_DIR / "gtsrb_cnn_clean.pth"
    torch.save({name: value.detach().cpu() for name, value in clean_model_gtsrb.state_dict().items()}, clean_checkpoint)
    clean_model_gtsrb = clean_model_gtsrb.to("cpu")
    del optimizer_clean_gtsrb
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print(f"Saved clean model weights to {clean_checkpoint.resolve()}")

    # Trojan training: selected source images carry trigger pixels and target labels.
    print("Training the model on the poisoned training loader...")
    trojaned_model_gtsrb = GTSRB_CNN(num_classes=NUM_CLASSES_GTSRB).to(device)
    optimizer_trojan_gtsrb = optim.Adam(
        trojaned_model_gtsrb.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    trojaned_losses_gtsrb = train_model(
        trojaned_model_gtsrb, trainloader_poisoned, criterion_gtsrb,
        optimizer_trojan_gtsrb, NUM_EPOCHS, device,
    )
    trojan_checkpoint = MODEL_OUTPUT_DIR / "gtsrb_cnn_trojaned.pth"
    torch.save({name: value.detach().cpu() for name, value in trojaned_model_gtsrb.state_dict().items()}, trojan_checkpoint)
    trojaned_model_gtsrb = trojaned_model_gtsrb.to("cpu")
    del optimizer_trojan_gtsrb
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print(f"Saved Trojan-training model weights to {trojan_checkpoint.resolve()}")
elif not RUN_TRAINING:
    print("Training is disabled. Set RUN_TRAINING = True to explicitly start both 20-epoch runs.")
else:
    print("Training skipped: required clean and attack data loaders are not all available.")

### COAE quick recall

- `learning_rate` scales optimizer update steps; an epoch is one full pass over the training set.
- Weight decay penalizes large weights and can trade off ordinary generalization against trigger strength.
- Training order: clear gradients → forward pass → compute cross-entropy → backpropagate → optimizer update.
- `y_hat = argmax(logits)` selects the class with the highest raw score.
- Accuracy is correct valid predictions divided by all valid examples.
- This HTB ASR uses triggered examples whose original label is `SOURCE_CLASS`; numerator counts those predicted as `TARGET_CLASS`.
- `-1` is an invalid-sample sentinel. Exclude it; if the denominator is empty, the metric is undefined.
- The clean model uses `trainloader_clean`; the Trojan-training model uses `trainloader_poisoned`. Their inputs/labels differ.
- `RUN_TRAINING` defaults to `False`; enabling it starts two potentially long training runs and saves weights under `model_outputs/`.

**Exam check:** If 45 of 50 triggered source-class images are predicted as the target, calculate source-only ASR. What does label `-1` mean? Why does the Trojan-training model use the poisoned loader while the baseline uses the clean loader?

### Section 18 scope check

Section 18 defines the training loop, clean accuracy evaluator, and source-only ASR function. The training cell is opt-in. Section 19 now uses those existing helpers to compare a clean baseline model with the Trojan-training model; it does not redefine the metric functions.


## Section 19 — Evaluating the Trojan Attack

This subsection compares two models under two input conditions. First, both models see the ordinary clean test set; this measures whether the Trojan-training process damaged normal classification. Second, both see triggered test images; the source-only ASR function checks whether triggered source-class images are redirected to the chosen target.

A backdoor experiment has two goals that pull in different directions: preserve useful ordinary behavior and learn the trigger-to-target behavior. A strong result therefore pairs high clean accuracy with high ASR for the Trojaned model, while the clean baseline should have low ASR.

The code below reuses `evaluate_model` and `calculate_asr_gtsrb` from Section 18. It loads or uses each checkpoint in turn, evaluates it, then moves it back to CPU before handling the other model. `RUN_EVALUATION` defaults to `False` so opening or running the notebook does not unexpectedly launch evaluation over the full test sets. This is a notebook safety choice; HTB demonstrates the evaluation calls directly.

### 19.1 Read the evaluation formulas

**Clean accuracy** is `100 × correct clean predictions / valid clean examples`. Say it as: “one hundred times the number of clean examples classified correctly, divided by the number of valid clean examples.” If 12,367 of 12,630 images are correct, then `100 × 12,367 / 12,630 ≈ 97.92%`.

**Source-only ASR** is `100 × triggered source images predicted as target / valid triggered source images`. Say it as: “one hundred times the number of triggered source images predicted as the target, divided by the number of valid triggered source images.” If 270 of 270 triggered Stop signs are predicted as Speed limit 60, then ASR is `100 × 270 / 270 = 100%`.

The denominator is important: this experiment counts only test images whose **original** label was the source class. It does not divide by every image in the triggered test loader. The triggered loader changes image pixels but keeps original labels so the evaluator can identify the source examples.

HTB's displayed example is:

| Model | Clean test accuracy | Source-only ASR | What it suggests |
| --- | ---: | ---: | --- |
| Clean baseline | 97.92% (12,367 / 12,630) | 0% (0 / 270) | Good normal performance; no observed trigger-to-target behavior. |
| Trojaned model | 97.55% (12,320 / 12,630) | 100% (270 / 270) | Normal accuracy stayed close to baseline while the trigger redirected every tested source image. |

These are the results printed in the HTB lesson, not results produced by this notebook run. A high ASR alone is not enough: a model that predicts the target for every input could have high ASR but terrible normal utility. Read ASR together with clean accuracy and the clean baseline's ASR.

In [ ]:
# Evaluation is opt-in. Enable only after the clean/triggered test loaders exist
# and trained weights are available either in memory or under MODEL_OUTPUT_DIR.
RUN_EVALUATION = False
EVALUATION_RESULTS = {}

# Check loader availability without assuming every earlier setup cell was run.
evaluation_loaders_ready = (
    "testloader_clean" in globals()
    and testloader_clean is not None
    and "testloader_triggered" in globals()
    and testloader_triggered is not None
)

if not RUN_EVALUATION:
    print("Evaluation is disabled. Set RUN_EVALUATION = True to compare both models.")
elif not evaluation_loaders_ready:
    print("Evaluation skipped: clean and triggered test loaders are both required.")
else:
    # The loss function is needed for average loss; define it if training was skipped.
    if "criterion_gtsrb" not in globals() or criterion_gtsrb is None:
        criterion_gtsrb = nn.CrossEntropyLoss()

    models_to_evaluate = [
        (
            "clean_baseline",
            globals().get("clean_model_gtsrb"),
            MODEL_OUTPUT_DIR / "gtsrb_cnn_clean.pth",
        ),
        (
            "trojaned_model",
            globals().get("trojaned_model_gtsrb"),
            MODEL_OUTPUT_DIR / "gtsrb_cnn_trojaned.pth",
        ),
    ]

    for model_name, model, checkpoint_path in models_to_evaluate:
        # If the model is not still in memory, restore its saved weights on CPU.
        if model is None and checkpoint_path.exists():
            model = GTSRB_CNN(num_classes=NUM_CLASSES_GTSRB)
            # This file should contain only a state_dict; restrict unpickling to weights.
            state_dict = torch.load(
                checkpoint_path, map_location="cpu", weights_only=True
            )
            model.load_state_dict(state_dict)

        if model is None:
            print(f"Skipping {model_name}: no model object or checkpoint was found.")
            continue

        print(f"\nEvaluating {model_name}...")
        model.to(device)
        try:
            # Clean images test ordinary task performance.
            clean_accuracy, clean_loss, _, _ = evaluate_model(
                model,
                testloader_clean,
                criterion_gtsrb,
                device,
                description=f"{model_name} on clean test images",
            )
            # Triggered inputs test the source-class backdoor behavior.
            source_asr = calculate_asr_gtsrb(
                model,
                testloader_triggered,
                SOURCE_CLASS,
                TARGET_CLASS,
                device,
            )
            EVALUATION_RESULTS[model_name] = {
                "clean_accuracy_percent": clean_accuracy,
                "clean_average_loss": clean_loss,
                "source_only_asr_percent": source_asr,
            }
        finally:
            # Only one model stays on the accelerator at a time.
            model.to("cpu")
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()

    print("\nComparison results:")
    for model_name, metrics in EVALUATION_RESULTS.items():
        print(
            f"{model_name}: clean accuracy={metrics['clean_accuracy_percent']:.2f}%, "
            f"source-only ASR={metrics['source_only_asr_percent']:.2f}%"
        )


### COAE quick recall — evaluating a backdoor

- **Accuracy:** correct clean predictions divided by all valid clean examples, shown as a percentage.
- **ASR:** triggered **source-class** examples predicted as **target** divided by all valid triggered source-class examples, shown as a percentage.
- **Read both together:** high Trojaned-model clean accuracy is a stealth signal; high Trojaned ASR is an effectiveness signal. Compare against the clean baseline.
- **Do not mix denominators:** this lesson's ASR denominator is source-only, not all triggered images.
- **Original labels matter:** triggered test images retain their original class labels so source examples can be filtered.
- `model.eval()` configures prediction behavior; `torch.no_grad()` avoids storing unneeded gradients.
- The notebook's evaluation gate defaults to `False`; set `RUN_EVALUATION = True` only when test loaders and trained models/checkpoints are ready.

**Exam check:** Why can clean accuracy look normal while a backdoor is still present? If 270 eligible triggered source images all become the target, what is the source-only ASR?

## Section 20 — Pickles and Tensor Steganography

This section discusses two different model-file risks. **Unsafe deserialization** can execute code while Python reconstructs a saved object. **Tensor steganography** hides information in the bit patterns of numerical model parameters. The first can provide an execution path; the second can provide a covert place to store data. Neither concept requires changing the model's neural-network architecture.

The security note matters to ordinary model use: a file extension such as `.pth` does not make a checkpoint safe. The notebook's Section 19 loader now explicitly requests `weights_only=True` because its checkpoints contain a tensor state dictionary. That restricted mode still does not prove that the model weights are benign; it only reduces the danger of object reconstruction.

### 20.1 Pickle and the checkpoint trust boundary

**Serialization** turns an in-memory object into bytes for storage. **Deserialization** reconstructs an object from those bytes. Python's `pickle` format can reconstruct complex objects, not just passive numbers. An object's `__reduce__` method can describe which callable and arguments should be used to rebuild it. If a malicious file is loaded by an unrestricted unpickler, those instructions may cause a callable to run during loading.

Say the risk chain aloud: **“untrusted file, object reconstruction, attacker-controlled reconstruction instructions, code execution.”** PyTorch's `torch.save` commonly uses pickle-based serialization, and `torch.load` reads such files. Do not load a checkpoint from an unknown source simply because its extension is `.pt` or `.pth`.

For ordinary parameter checkpoints, prefer a state dictionary and explicitly request `torch.load(path, map_location="cpu", weights_only=True)` when the installed PyTorch version supports it. This restricts reconstruction to the limited types needed for weights, such as tensors and basic containers. Safetensors is another tensor-oriented format that avoids Python pickle object reconstruction. Use trusted sources and verify file provenance; signatures or hashes can help establish provenance only when the expected signature/hash itself comes from a trusted channel.

**Separate two questions:** restricted loading addresses whether loading can execute arbitrary Python reconstruction behavior. It does not establish that the model's predictions are safe or honest. A safely loaded state dictionary could still encode a backdoor. Serialization safety and model-behavior review are different controls.

### 20.2 Tensors, float32, and the bit-level idea

A **tensor** is a multi-dimensional array. A vector is a one-dimensional tensor; a matrix is two-dimensional; image batches and convolution filters can have more dimensions. A PyTorch **state dictionary** (`state_dict`) is a mapping from parameter names to tensors containing the model's learned weights and biases.

A common 32-bit floating-point format divides one value into 1 sign bit, 8 exponent bits, and 23 mantissa bits. For a normalized number, the course formula is:

`value = (-1)^s × (1.m) × 2^(E_stored - bias)`

Read it aloud: **“negative one to the power s, times one point m, times two to the power stored E minus bias.”** `s` is the sign bit; `m` is the fraction encoded by mantissa bits; and `E_stored - bias` is the actual exponent. For float32, the exponent bias is 127. The leading `1` in `1.m` is implicit for ordinary normalized values. (Zero, infinity, and subnormal values have special encodings.)

The lesson's example is `0.15625` in decimal:

`0.15625₁₀ = 0.00101₂ = 1.01₂ × 2⁻³`

Read the subscripts as **“base ten”** and **“base two.”** `1.01₂ × 2⁻³` means move the binary point three positions left from `1.01₂`, yielding `0.00101₂`. The sign is 0, actual exponent is -3, and stored exponent is `-3 + 127 = 124` (`01111100` in binary). Mantissa begins `01`, followed by zero padding to 23 bits.

#### Why low-order bits change the value less

For this example, the last mantissa bit contributes `2⁻²⁶ ≈ 1.49 × 10⁻⁸`. Say this as **“two to the negative twenty-sixth, approximately one point four nine times ten to the negative eighth.”** Flipping that bit changes `0.15625` to about `0.1562500149`.

The leftmost mantissa bit contributes `2⁻⁴ = 0.0625`; flipping it changes the value to `0.21875`. The more significant bit has a much larger effect. This is why hiding information in low-order mantissa bits can make only a tiny numerical difference, while changing higher-order bits is more likely to disturb the model's behavior.

**Steganography** hides the existence of a message. **Encryption** makes a message unreadable without a key. Tensor steganography treats parameter bits as a carrier; the section's example focuses on the numerical effect of flipping bits, not on creating a malicious file.

In [ ]:
# Safe representation demo: inspect IEEE-754 bits without loading any pickle or model file.
# struct fixes the example to a 32-bit float and uses big-endian ordering for readable bits.
import struct

FLOAT32_EXAMPLE = 0.15625

def float32_to_bits(value):
    """Return the 32 IEEE-754 bits of value as a string, most significant bit first."""
    packed_float = struct.pack(">f", value)
    integer_bits = struct.unpack(">I", packed_float)[0]
    return format(integer_bits, "032b")

def bits_to_float32(bit_string):
    """Interpret a 32-character bit string as an IEEE-754 float32 value."""
    integer_bits = int(bit_string, 2)
    packed_bits = struct.pack(">I", integer_bits)
    return struct.unpack(">f", packed_bits)[0]

def flip_bit(bit_string, string_index):
    """Return a copy with one character (0 or 1) flipped."""
    old_bit = bit_string[string_index]
    new_bit = "1" if old_bit == "0" else "0"
    return bit_string[:string_index] + new_bit + bit_string[string_index + 1:]

original_bits = float32_to_bits(FLOAT32_EXAMPLE)
# Bit layout in this string: sign at index 0, exponent at 1..8, mantissa at 9..31.
lsb_flipped_bits = flip_bit(original_bits, 31)          # Mantissa bit 0, least significant.
mantissa_msb_flipped_bits = flip_bit(original_bits, 9)  # Mantissa bit 22, most significant.

lsb_flipped_value = bits_to_float32(lsb_flipped_bits)
mantissa_msb_flipped_value = bits_to_float32(mantissa_msb_flipped_bits)

print(f"Original float32:       {FLOAT32_EXAMPLE}")
print(f"Original bits:          {original_bits}")
print(f"LSB-flipped value:      {lsb_flipped_value:.12f}")
print(f"LSB change:             {lsb_flipped_value - FLOAT32_EXAMPLE:.12g}")
print(f"Mantissa-MSB value:     {mantissa_msb_flipped_value:.8f}")
print(f"Mantissa-MSB change:    {mantissa_msb_flipped_value - FLOAT32_EXAMPLE:.8f}")


### COAE quick recall — model-file security

- Pickle is a serialization format that can reconstruct Python objects; unrestricted loading of an untrusted pickle can execute code during reconstruction.
- `__reduce__` can describe a callable and arguments for reconstruction; dangerous instructions in an untrusted file create the risk.
- `.pth` and `.pt` are filename extensions, not safety guarantees.
- Prefer state dictionaries; explicitly use `weights_only=True` with `torch.load` where supported, or use a tensor-only format such as Safetensors.
- Safer loading does **not** prove model behavior is benign; a backdoor can live in weights without unsafe deserialization.
- Float32 layout: 1 sign bit, 8 exponent bits, 23 mantissa bits. Low-order mantissa changes are much smaller than high-order changes.
- Steganography hides that information exists; encryption hides what information says.

**Exam check:** Which risk concerns execution during file loading, and which concerns hidden data in parameters? Why does `weights_only=True` help with the first without proving the model is free of a backdoor?

## Section 21 — Training SimpleNet and Measuring Raw Capacity

This subsection creates a small, benign PyTorch network as the model whose parameter storage we will study. The active prediction path uses two linear layers and ReLU. A deliberately unused `large_layer` supplies a larger named tensor in the `state_dict`; the course uses it to calculate how many low-order bits it could hold in a later demonstration.

The training data is synthetic: 100 rows of 10 random features and a noisy linear target. The model trains with mean squared error and Adam, then saves only its parameter `state_dict`. The notebook keeps this tiny training and save operation opt-in. The capacity calculation itself runs immediately and does not create or load a checkpoint.

### 21.1 Capacity math, read aloud

Let `N` be the number of values in a chosen tensor (`tensor.numel()`), and `n` be the number of low-order bits used from each value.

`Capacity_bits = N × n`

Read it as: **“capacity in bits equals N times n.”** If a tensor has 20,480 values and we allocate 2 bits per value, that gives `20,480 × 2 = 40,960 bits`.

`Capacity_bytes = floor((N × n) / 8)`

Read `floor` as **“round down to a whole number.”** There are 8 bits in one byte, so `40,960 / 8 = 5,120 bytes`. That is 5 KiB (5,120 bytes), before any extra space needed for a real message format. With 1 bit per value, the same tensor provides `20,480 bits / 8 = 2,560 bytes`.

Remember: `numel()` counts numeric values, not bytes. For float32, each value normally takes 32 bits, so 20,480 float32 values occupy 81,920 bytes in memory. The lesson's capacity formula counts only the selected `n` low-order bits per value.

In [ ]:
# SimpleNet and its synthetic training example use CPU and a tiny dataset.
# This keeps the course demonstration lightweight and independent of GPU memory.
from torch.utils.data import TensorDataset, DataLoader

SEED = 1337
np.random.seed(SEED)
torch.manual_seed(SEED)

class SimpleNet(nn.Module):
    """Small regression network with an extra stored layer for capacity analysis."""

    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        # Registered in state_dict, but intentionally not used by forward().
        self.large_layer = nn.Linear(hidden_size, hidden_size * 5)

    def forward(self, x):
        x = self.fc1(x)      # Combine the 10 input features into hidden activations.
        x = self.relu(x)     # Keep positive activations and set negative ones to zero.
        return self.fc2(x)   # Produce one numeric prediction per row.

input_dim = 10
hidden_dim = 64
output_dim = 1
simple_net = SimpleNet(input_dim, hidden_dim, output_dim).cpu()

print("SimpleNet architecture:")
print(simple_net)
print("\nParameter tensors saved in its state_dict:")
for name, tensor in simple_net.state_dict().items():
    print(f"{name}: shape={tuple(tensor.shape)}, elements={tensor.numel()}, dtype={tensor.dtype}")

# The capacity calculation is raw bits available in this one selected tensor.
capacity_tensor = simple_net.state_dict()["large_layer.weight"]
N = capacity_tensor.numel()  # Number of floating-point values in the tensor.
for n in (1, 2):             # Compare using one or two low-order bits per value.
    capacity_bits = N * n
    capacity_bytes = capacity_bits // 8  # Integer division is the floor operation.
    print(
        f"With {n} bit(s) per value: {capacity_bits:,} bits, "
        f"{capacity_bytes:,} bytes"
    )

# Optional: train on generated data and save a parameter-only state_dict.
# This takes only a few CPU batches and performs no network access.
RUN_SIMPLENET_TRAINING = False
if RUN_SIMPLENET_TRAINING:
    generator = torch.Generator().manual_seed(SEED)
    X_train = torch.randn(100, input_dim, generator=generator)
    true_weights = torch.randn(input_dim, output_dim, generator=generator)
    noise = 0.5 * torch.randn(100, output_dim, generator=generator)
    y_train = X_train @ true_weights + noise

    dataloader = DataLoader(
        TensorDataset(X_train, y_train), batch_size=16, shuffle=False
    )
    criterion = nn.MSELoss()
    optimizer = optim.Adam(simple_net.parameters(), lr=0.01)
    num_epochs = 5
    simple_net.train()

    for epoch in range(num_epochs):
        weighted_loss_sum = 0.0
        sample_count = 0
        for inputs, targets in dataloader:
            optimizer.zero_grad(set_to_none=True)
            predictions = simple_net(inputs)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            batch_size = inputs.size(0)
            weighted_loss_sum += loss.item() * batch_size
            sample_count += batch_size
        print(
            f"Epoch {epoch + 1}/{num_epochs}: "
            f"sample-weighted MSE={weighted_loss_sum / sample_count:.4f}"
        )

    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    simple_net_checkpoint = MODEL_OUTPUT_DIR / "simplenet_target_state_dict.pth"
    torch.save(simple_net.state_dict(), simple_net_checkpoint)
    print(f"Saved parameter state_dict to {simple_net_checkpoint.resolve()}")
else:
    print("Synthetic training is disabled; enable RUN_SIMPLENET_TRAINING to run it.")


### COAE quick recall — model capacity

- A PyTorch `state_dict` stores named parameter tensors; `tensor.numel()` is the number of values in one tensor.
- Raw capacity bits = `N × n`, with `N` values and `n` chosen bits per value.
- Raw capacity bytes = `floor((N × n) / 8)`.
- `large_layer.weight` has shape 320 × 64 = 20,480 values. At 2 bits each: 40,960 bits = 5,120 bytes = 5 KiB.
- `large_layer` is registered and saved but not used in the forward pass, so it does not affect predictions in this example.
- The capacity is theoretical; it does not establish that a payload is present or that the model is safe.
- This notebook's synthetic training is CPU-only and disabled by default. Saving writes only a state dictionary; it does not load or execute an external checkpoint.

**Exam check:** For 12,000 values and 2 bits per value, how many raw bytes are available? Why is that not the same as the tensor's total float32 size?

## Section 22 — Steganography Tools: encode and decode tensor bits

HTB introduces `encode_lsb` and `decode_lsb`: the encoder stores a byte message in the least significant bits (LSBs) of float32 tensor values, and the decoder reverses the process. The message begins with a four-byte, big-endian length so decoding knows exactly how many payload bytes to recover. This is steganography (concealing presence), not encryption (concealing meaning).

The implementation below keeps HTB’s format and bit order. It also uses a single bit cursor when decoding. That matters when `num_lsb` does not divide the 32-bit length header evenly: for example, with three bits per float, the header ends partway through a float’s three-bit group, so the remaining bit must be the first payload bit rather than discarded.

The code expects a `torch.float32` tensor, returns a detached clone, and does not mutate the source tensor. The Python loop is intentionally explicit for learning; for very large tensors, a vectorized implementation would be faster.

### 22.1 Formulas and pronunciation

`capacity_bits = N × n` — read: **“capacity in bits equals N times n.”** `N` is the number of float values; `n` is the LSB count used per value.

`bits_needed = 8 × (4 + P)` — read: **“bits needed equals eight times four plus P.”** `P` is the payload length in bytes, `4` is the four-byte length header, and `8` converts bytes to bits.

The mask is `(1 << n) - 1` — read: **“one shifted left by n, minus one.”** For `n = 2`, this is binary `11`, selecting the two lowest bits. Encoding uses `(old_bits AND NOT mask) OR message_bits`: clear the chosen low bits, then insert the next message-bit group.

Example: a 20,480-value tensor at two bits per value holds 40,960 bits. A two-byte message plus its four-byte header needs `8 × (4 + 2) = 48` bits, so it consumes 24 float values. For `n = 3`, the 32-bit header uses 10 full groups (30 bits) and two bits of the next group; decoding must keep the third bit as the first payload bit.

In [ ]:
import struct
import torch

def encode_lsb(tensor_orig: torch.Tensor, data_bytes: bytes, num_lsb: int = 2) -> torch.Tensor:
    """Return a clone with a length-prefixed byte message stored in float32 LSBs."""
    if tensor_orig.dtype != torch.float32:
        raise TypeError("Tensor must be float32.")
    if not isinstance(data_bytes, bytes):
        raise TypeError("data_bytes must be a bytes object.")
    if not isinstance(num_lsb, int) or not 1 <= num_lsb <= 8:
        raise ValueError("num_lsb must be an integer from 1 through 8.")
    if len(data_bytes) > 0xFFFFFFFF:
        raise ValueError("The four-byte length header supports at most 2^32-1 payload bytes.")

    # Clone/detach so encoding does not change the caller’s tensor or autograd history.
    tensor = tensor_orig.clone().detach()
    flat = tensor.view(-1)  # The clone is contiguous; flattening eases sequential storage.
    data_to_embed = struct.pack(">I", len(data_bytes)) + data_bytes  # Length prefix, then payload.
    total_bits_needed = len(data_to_embed) * 8  # Every byte contributes eight bits.
    capacity_bits = flat.numel() * num_lsb  # N × n from the capacity formula.
    if total_bits_needed > capacity_bits:
        raise ValueError(f"Tensor too small: needs {total_bits_needed} bits, has {capacity_bits} bits.")

    mask = (1 << num_lsb) - 1  # n ones in binary; selects the n lowest bits.
    bit_cursor = 0  # Next message bit, MSB-first within each byte.
    elements_used = (total_bits_needed + num_lsb - 1) // num_lsb  # Ceiling division.
    for element_index in range(elements_used):
        original_float = float(flat[element_index].item())
        # Convert the float32 storage to its 32-bit unsigned integer bit pattern.
        old_bits = struct.unpack(">I", struct.pack(">f", original_float))[0]
        message_group = 0
        for group_position in range(num_lsb):
            if bit_cursor >= total_bits_needed:  # Final group may be only partly filled.
                # Keep present bits at the group's high end; decoder reads high-to-low.
                message_group <<= num_lsb - group_position
                break
            byte_index, bit_in_byte = divmod(bit_cursor, 8)
            message_bit = (data_to_embed[byte_index] >> (7 - bit_in_byte)) & 1
            message_group = (message_group << 1) | message_bit  # Append the next bit.
            bit_cursor += 1

        # Clear the selected low bits, then insert this message group.
        new_bits = (old_bits & (~mask & 0xFFFFFFFF)) | message_group
        new_float = struct.unpack(">f", struct.pack(">I", new_bits))[0]
        flat[element_index] = new_float

    print(f"Encoded {total_bits_needed} bits into {elements_used} tensor elements.")
    return tensor


def decode_lsb(tensor_modified: torch.Tensor, num_lsb: int = 2) -> bytes:
    """Recover the length-prefixed byte message from float32 tensor LSBs."""
    if tensor_modified.dtype != torch.float32:
        raise TypeError("Tensor must be float32.")
    if not isinstance(num_lsb, int) or not 1 <= num_lsb <= 8:
        raise ValueError("num_lsb must be an integer from 1 through 8.")

    flat = tensor_modified.detach().view(-1)
    capacity_bits = flat.numel() * num_lsb
    mask = (1 << num_lsb) - 1
    bit_cursor = 0  # Shared by header and payload so the bit stream stays aligned.

    def read_bit() -> int:
        nonlocal bit_cursor
        if bit_cursor >= capacity_bits:
            raise ValueError("Tensor ended before the requested message bits were available.")
        element_index, position_in_group = divmod(bit_cursor, num_lsb)
        value = float(flat[element_index].item())
        stored_bits = struct.unpack(">I", struct.pack(">f", value))[0]
        group = stored_bits & mask
        # Read each group high-to-low, matching encode_lsb.
        bit = (group >> (num_lsb - 1 - position_in_group)) & 1
        bit_cursor += 1
        return bit

    def read_bytes(byte_count: int) -> bytes:
        result = bytearray()
        for _ in range(byte_count):
            value = 0
            for _ in range(8):
                value = (value << 1) | read_bit()  # Shift each bit into the byte.
            result.append(value)
        return bytes(result)

    if capacity_bits < 32:
        raise ValueError("Tensor is too small even for the four-byte length header.")
    payload_length = struct.unpack(">I", read_bytes(4))[0]  # Decode the 32-bit length.
    payload_bits_needed = payload_length * 8
    if bit_cursor + payload_bits_needed > capacity_bits:
        raise ValueError("Length header claims a payload larger than the remaining tensor capacity.")
    return read_bytes(payload_length)

### 22.2 Round-trip demonstration

The message below is harmless. We encode it into a synthetic float32 tensor, decode it, and compare the recovered bytes with the original. The final line measures the largest numerical change to any float; low-order bit changes are usually small, but that is a measurement, not a promise of zero effect on a model.

In [ ]:
original_tensor = torch.linspace(-1.0, 1.0, steps=128, dtype=torch.float32)
message = b"HTB"
encoded_tensor = encode_lsb(original_tensor, message, num_lsb=2)
recovered_message = decode_lsb(encoded_tensor, num_lsb=2)

print("Recovered message:", recovered_message)
print("Round trip matched:", recovered_message == message)
print("Original tensor unchanged:", torch.equal(original_tensor, torch.linspace(-1.0, 1.0, steps=128)))
print("Maximum absolute value change:", (encoded_tensor - original_tensor).abs().max().item())

### COAE quick recall — tensor LSB steganography

- Capacity: `N × n` bits. A four-byte header plus `P` payload bytes needs `8 × (4 + P)` bits.
- `struct.pack(">f", value)` turns a number into four big-endian float32 bytes; `struct.unpack(">I", ...)` reads those bits as an unsigned integer.
- Mask `(1 << n) - 1` selects the bottom `n` bits; clear them, then OR in the message group.
- The decoder must use the same `num_lsb` and bit order, read the header first, and preserve cursor position across the header/payload boundary.
- This is a teaching implementation, not a secure transport or guarantee of imperceptibility. More LSBs provide more capacity but can produce more numerical distortion.

**Exam check:** Why does the encoder store a length prefix? If `num_lsb = 3`, why must decoding preserve the unused part of the float group where the 32-bit length ends?

## Section 23 — The Attack: connect the payload, tensor, and pickle trigger

HTB combines the previous building blocks into one attack chain: prepare payload source as UTF-8 bytes; embed those bytes in `large_layer.weight`; replace that parameter in a copied `state_dict`; then put the modified state dictionary inside an object whose pickle reconstruction hook (`__reduce__`, pronounced “dunder reduce”) supplies loader instructions.

The lesson's payload is a reverse shell: the target initiates a network connection back to a listener. Its listener IP and port must be reachable from the target. The HTB code then redirects the shell process's standard input/output/error to that connection. The live course payload is executable and can provide remote command access. This study notebook uses a harmless print string instead; its example does not open a socket, spawn a shell, save a Trojan file, or unpickle crafted data.

**Timing matters:** `__reduce__` is called while the wrapper is serialized so pickle can record the reconstruction recipe. The callable and arguments in that recipe run later when an unsafe unpickler reconstructs the wrapper. The trigger is deserialization, not merely saving the pickle.

### 23.1 Payload size and attack sequence

Let `P` be payload size in bytes, `n` be low-order bits used per float, and `N` be the number of values in the chosen tensor.

`bits_needed = 8 × (4 + P)` — read aloud: **“bits needed equals eight times four plus P.”** The four is the length prefix; `P` is the message; eight converts bytes to bits.

`elements_needed = ceil(bits_needed / n)` — read aloud: **“elements needed equals the ceiling of bits needed divided by n.”** `ceil` means round up to a whole tensor element. The equivalent integer expression is `(bits_needed + n - 1) // n`. Example: 112 bits with `n = 3` needs `ceil(112/3) = 38` tensor elements.

The chain is: payload source → UTF-8 bytes → LSB-encoded tensor → copied `state_dict` → wrapper pickle recipe → unsafe load invokes recipe → loader extracts and runs payload. The carrier capacity is `N × n` bits; the four-byte header is part of the data that must fit.

In [ ]:
import pickle
import pickletools
import torch

# Use only harmless source text for this study example. It is stored as bytes,
# but this notebook never executes it.
payload_code_string = "print('Harmless demonstration: decoded source was not executed')"
payload_bytes = payload_code_string.encode("utf-8")
print(f"Harmless payload length: {len(payload_bytes)} bytes")

# Recreate the course's SimpleNet so this subsection does not rely on saved files.
demo_model = SimpleNet(input_size=10, hidden_size=64, output_size=1).cpu()
demo_state_dict = dict(demo_model.state_dict())  # Copy the mapping; tensors are not edited in place.
target_key = "large_layer.weight"
original_tensor = demo_state_dict[target_key]
NUM_LSB = 2

# Formula: N × n bits of capacity; 8 × (4 + P) bits required including the header.
capacity_bits = original_tensor.numel() * NUM_LSB
bits_needed = 8 * (4 + len(payload_bytes))
elements_needed = (bits_needed + NUM_LSB - 1) // NUM_LSB  # Ceiling division.
print(f"Capacity: {capacity_bits:,} bits")
print(f"Payload plus length header: {bits_needed:,} bits")
print(f"Elements needed: {elements_needed:,}; available: {original_tensor.numel():,}")
if elements_needed > original_tensor.numel():
    raise ValueError("The demonstration payload does not fit in the selected tensor.")

# Encode into a clone and replace only the selected tensor in the copied mapping.
modified_tensor = encode_lsb(original_tensor, payload_bytes, NUM_LSB)
demo_state_dict[target_key] = modified_tensor
recovered_bytes = decode_lsb(demo_state_dict[target_key], NUM_LSB)
print("Payload bytes round-trip correctly:", recovered_bytes == payload_bytes)
print("Original model tensor remains unchanged:", torch.equal(demo_model.state_dict()[target_key], original_tensor))

# Show pickle's reconstruction recipe with a harmless callable. pickle.dumps()
# invokes __reduce__ to record this recipe; we intentionally do NOT call pickle.loads().
class HarmlessReduceDemo:
    def __reduce__(self):
        return (print, ("This message would print only if the pickle were loaded.",))

serialized_recipe = pickle.dumps(HarmlessReduceDemo())
print("
Pickle opcodes for the harmless reconstruction recipe:")
pickletools.dis(serialized_recipe)
print("
No pickle.loads() call was made; the stored callable was not invoked.")

### COAE quick recall — Trojan model attack chain

- `P` is payload bytes, `n` is LSBs per float, and `N` is tensor values.
- Required bits = `8 × (4 + P)`; required elements = `ceil(required_bits / n)`.
- `__reduce__` (“dunder reduce”) provides the callable and arguments pickle should use to reconstruct an object.
- Serialization records the recipe; unsafe deserialization invokes it. Saving alone is not the trigger.
- A reverse shell calls from target back to listener; the listener address must be reachable from the target.
- Embedding code in parameters and executing code during deserialization are separate stages.
- The notebook demo uses inert source and a harmless `print` callable, never `pickle.loads`, never a socket, and never a shell.
- Defend by avoiding unsafe loads of untrusted artifacts, using restricted tensor-only loading where supported, and verifying artifact provenance/integrity.

**Exam check:** What runs when the pickle is created, and what runs when it is loaded? Why does the listener address need to be reachable from the target?